# Task 08: CatBoost pointwise baseline

Read-only analysis of a completed task08 artifact. Training and candidate generation are never started from this notebook.

In [ ]:
import json
import os
from pathlib import Path

import polars as pl
from IPython.display import display

from rankers import CatBoostPointwiseModel

artifact = Path(os.environ.get("TASK08_ARTIFACT", "artifacts/task08_catboost_pointwise_v1"))
if not artifact.is_dir():
    raise FileNotFoundError(f"Run task08 first or set TASK08_ARTIFACT: {artifact}")
config = json.loads((artifact / "config.json").read_text())
metrics = json.loads((artifact / "metrics.json").read_text())
print(artifact, config["mode"], metrics["tree_count"], metrics["best_iteration"])

In [ ]:
canonical = metrics["canonical"]
comparison = pl.DataFrame([
    {"ranker": name, **values}
    for name, values in canonical["rankers"].items()
]).select(
    "ranker", "precision_at_20_all_targets",
    "precision_at_20_labeled_users", "final_hits"
).sort("precision_at_20_all_targets", descending=True)
display(comparison)
display(pl.DataFrame([
    {"candidate_set": name, **values}
    for name, values in canonical["candidate_sets"].items()
]))

In [ ]:
importance = pl.read_parquet(artifact / "feature_importance.parquet")
display(importance.head(30))

In [ ]:
model = CatBoostPointwiseModel.from_artifact(artifact / "model")
assert model.tree_count == metrics["tree_count"]
assert model.best_iteration == metrics["best_iteration"]
print(model.get_config())